In [ ]:
pip install fastapi langchain langchain-chroma langchain-ollama langchain-community starlette uvicorn nest_asyncio

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 设置PDF文件路径
pdf_path = "../../files/pdf/en/Transformer.pdf"

# 加载PDF文档并分割文本
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# 使用递归文本切分器来切分文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

In [4]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

# 存储分割后的文档到向量数据库
vectorstore = Chroma.from_documents(documents=splits, embedding=OllamaEmbeddings(model="nomic-embed-text"))

In [5]:
# 构建检索器
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama

# 定义RAG提示模板
prompt = hub.pull("rlm/rag-prompt")

# 格式化检索到的文档
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 定义RAG链
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatOllama(model="deepseek-r1:7b")
    | StrOutputParser()
)

In [7]:
import json

# 生成答案函数
async def generate_answer(question: str):
    response = await rag_chain.ainvoke(question)
    return response

# 生成流式响应
async def generate_streaming_response(question: str):
    async for chunk in rag_chain.astream(question):  # 使用astream逐块获取响应
        yield json.dumps({"answer chunk": chunk}) + "\n"  # 按流式返回每一块内容

In [8]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from starlette.responses import StreamingResponse

# 创建FastAPI应用
app = FastAPI()

# 定义输入模型
class QueryModel(BaseModel):
    question: str
    stream: bool = False  # 默认不流式返回

# 创建POST路由处理查询
@app.post("/query/")
async def query_question(query: QueryModel):
    try:
        if query.stream:
            # 如果stream为True，使用流式响应
            return StreamingResponse(generate_streaming_response(query.question), media_type="text/json")
        else:
            # 否则直接返回完整答案
            answer = await generate_answer(query.question)  # 使用await获取完整的答案
            return {"answer": answer}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [9]:
import nest_asyncio
import uvicorn
nest_asyncio.apply()
uvicorn.run(app, host="127.0.0.1", port=8000)

INFO:     Started server process [2000]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:62498 - "POST /query/ HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [2000]


In [10]:
vectorstore.delete_collection()